<a href="https://colab.research.google.com/github/Maria-lin/F1-Analytics/blob/main/detection_anomalies_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏧 Détection d'Anomalies sur le Réseau GAB
## Identification automatique des comportements atypiques sur les Guichets Automatiques Bancaires

---

> **Destinataires :** Experts métier – Responsables réseau GAB  
> **Objectif :** Identifier automatiquement les GAB au comportement inhabituel, sans étiquette préalable  
> **Approche :** Détection d'anomalies non supervisée (Isolation Forest)

---

## 1. 🏦 Introduction Métier

### Pourquoi surveiller le comportement des GAB ?

Un **Guichet Automatique Bancaire (GAB)** est bien plus qu'une simple machine à billets.  
Chaque jour, il enregistre des centaines d'opérations : retraits, refus, captures de carte…  
L'ensemble de ces comportements forme une **empreinte** propre à chaque automate.

**Un GAB "normal"** présente :
- Un volume de retraits stable, cohérent avec son emplacement (centre-ville vs zone rurale)
- Un taux de capture de carte bas (les cartes capturées indiquent des incidents)
- Des montants moyens dans une plage raisonnable
- Une activité rythmée par la saisonnalité (fêtes, été…)

**Un GAB "atypique"** peut révéler :
- 🔴 Un problème technique (pannes, captures anormalement fréquentes)
- 🔴 Une fraude en cours (montants suspects, activité nocturne inhabituelle)
- 🔴 Un dysfonctionnement réseau (cartes d'un seul type concentrées sur un automate)
- 🔴 Un comportement client inhabituel (retraits massifs, activité hors horaires)

### Pourquoi un modèle ?

> *« Les anomalies évidentes, on les voit déjà. »*

C'est vrai. Mais le modèle détecte ce que l'œil humain ne peut pas voir :
- Des **combinaisons** subtiles de signaux faibles (montant moyen légèrement élevé + taux capture légèrement haut + activité nocturne légèrement supérieure = anomalie)
- Des anomalies sur **100+ variables simultanément**
- Une **surveillance continue** sur l'ensemble du parc, sans fatigue

---

## 2. ⚙️ Imports et Configuration

In [ ]:
# ── Librairies standards ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Visualisation ──────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import folium
from folium.plugins import MarkerCluster, HeatMap

# ── Machine Learning ───────────────────────────────────────────────────────────
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

# ── Style global ───────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
COULEUR_NORMAL   = '#2196F3'   # bleu
COULEUR_ANOMALIE = '#F44336'   # rouge
COULEUR_ACCENT   = '#FF9800'   # orange

print('✅ Librairies chargées avec succès.')

## 3. 📂 Chargement des Données

Le dataset **`fiche_identite_gab`** contient une ligne par GAB et par mois.  
Chaque ligne résume l'activité complète de l'automate sur la période.

In [ ]:
# ── Chargement du dataset ─────────────────────────────────────────────────────
# Adapter le chemin selon votre environnement Dataiku ou local
# df = pd.read_csv('fiche_identite_gab.csv')
# df = dataiku.Dataset('fiche_identite_gab').get_dataframe()  # Dataiku

# ── Simulation de données réalistes pour démonstration ───────────────────────
np.random.seed(42)
n_normal   = 450
n_anomalie = 30
N = n_normal + n_anomalie

periodes = pd.date_range('2024-01-01', periods=12, freq='MS')

def generer_dataset(n_normal=450, n_anomalie=30):
    # GAB normaux
    df_normal = pd.DataFrame({
        'num_automate'        : [f'GAB_{i:04d}' for i in range(n_normal)],
        'annee'               : np.random.choice([2024, 2025], n_normal),
        'mois'                : np.random.randint(1, 13, n_normal),
        'ret_nb'              : np.random.normal(800,  150, n_normal).clip(100),
        'ret_montant_total'   : np.random.normal(120000, 25000, n_normal).clip(5000),
        'ret_montant_moyen'   : np.random.normal(150, 30, n_normal).clip(50),
        'ret_montant_max'     : np.random.normal(500, 80, n_normal).clip(200),
        'ret_montant_stddev'  : np.random.normal(60, 15, n_normal).clip(5),
        'ret_nb_nuit'         : np.random.normal(80, 25, n_normal).clip(0),
        'ret_nb_weekend'      : np.random.normal(180, 40, n_normal).clip(0),
        'ret_pct_nuit'        : np.random.normal(10, 3, n_normal).clip(0, 100),
        'ret_pct_weekend'     : np.random.normal(22, 5, n_normal).clip(0, 100),
        'cap_nb'              : np.random.poisson(4, n_normal),
        'taux_capture_pct'    : np.random.normal(0.5, 0.3, n_normal).clip(0),
        'nb_ope_reseau_franfinance'  : np.random.normal(30, 10, n_normal).clip(0),
        'nb_ope_reseau_cb'  : np.random.normal(350, 60, n_normal).clip(0),
        'nb_ope_reseau_trionis'  : np.random.normal(15, 8, n_normal).clip(0),
        'nb_ope_reseau_ppl'  : np.random.normal(12, 6, n_normal).clip(0),
        'nb_ope_reseau_mastercard'  : np.random.normal(120, 30, n_normal).clip(0),
        'nb_ope_reseau_configona'  : np.random.normal(8, 4, n_normal).clip(0),
        'nb_ope_reseau_interne'  : np.random.normal(80, 20, n_normal).clip(0),
        'nb_ope_reseau_casino'  : np.random.normal(10, 5, n_normal).clip(0),
        'nb_ope_reseau_accord'  : np.random.normal(25, 10, n_normal).clip(0),
        'nb_ope_reseau_visa'  : np.random.normal(180, 40, n_normal).clip(0),
        'nb_ope_reseau_cos'  : np.random.normal(6, 3, n_normal).clip(0),
        'nb_ope_reseau_jcb'  : np.random.normal(4, 3, n_normal).clip(0),
        'nb_ope_reseau_postepargne'  : np.random.normal(5, 3, n_normal).clip(0),
        'nb_ope_reseau_diners_et_discovery'  : np.random.normal(3, 2, n_normal).clip(0),
        'nb_ope_reseau_autres'  : np.random.normal(10, 5, n_normal).clip(0),
        'type_gab_e_i'        : np.random.choice(['Interne', 'Externe'], n_normal, p=[0.4, 0.6]),
        'code_postal'         : np.random.choice(['75001','69001','13001','33000','31000','59000'], n_normal),
        'longitude'           : np.random.uniform(-4.5, 8.2, n_normal),
        'latitude'            : np.random.uniform(42.3, 51.1, n_normal),
        'is_anomalie_reelle'  : 0
    })

    # GAB atypiques (anomalies injectées)
    df_anomalie = pd.DataFrame({
        'num_automate'        : [f'GAB_ANOM_{i:03d}' for i in range(n_anomalie)],
        'annee'               : np.random.choice([2024, 2025], n_anomalie),
        'mois'                : np.random.randint(1, 13, n_anomalie),
        'ret_nb'              : np.random.choice(
                                    [np.random.normal(1800, 200, n_anomalie//2),
                                     np.random.normal(100,   30, n_anomalie//2)],
                                    replace=False).flatten().clip(0),
        'ret_montant_total'   : np.random.normal(350000, 60000, n_anomalie).clip(1000),
        'ret_montant_moyen'   : np.random.normal(420, 80, n_anomalie).clip(100),
        'ret_montant_max'     : np.random.normal(1200, 200, n_anomalie).clip(300),
        'ret_montant_stddev'  : np.random.normal(180, 40, n_anomalie).clip(5),
        'ret_nb_nuit'         : np.random.normal(250, 60, n_anomalie).clip(0),
        'ret_nb_weekend'      : np.random.normal(500, 80, n_anomalie).clip(0),
        'ret_pct_nuit'        : np.random.normal(35, 8, n_anomalie).clip(0, 100),
        'ret_pct_weekend'     : np.random.normal(45, 8, n_anomalie).clip(0, 100),
        'cap_nb'              : np.random.poisson(25, n_anomalie),
        'taux_capture_pct'    : np.random.normal(8.5, 2.5, n_anomalie).clip(0),
        'nb_ope_reseau_franfinance'  : np.random.normal(400, 80, n_anomalie).clip(0),
        'nb_ope_reseau_cb'  : np.random.normal(20, 10, n_anomalie).clip(0),
        'nb_ope_reseau_trionis'  : np.random.normal(15, 8, n_anomalie).clip(0),
        'nb_ope_reseau_ppl'  : np.random.normal(12, 6, n_anomalie).clip(0),
        'nb_ope_reseau_mastercard'  : np.random.normal(120, 30, n_anomalie).clip(0),
        'nb_ope_reseau_configona'  : np.random.normal(8, 4, n_anomalie).clip(0),
        'nb_ope_reseau_interne'  : np.random.normal(80, 20, n_anomalie).clip(0),
        'nb_ope_reseau_casino'  : np.random.normal(10, 5, n_anomalie).clip(0),
        'nb_ope_reseau_accord'  : np.random.normal(25, 10, n_anomalie).clip(0),
        'nb_ope_reseau_visa'  : np.random.normal(180, 40, n_anomalie).clip(0),
        'nb_ope_reseau_cos'  : np.random.normal(6, 3, n_anomalie).clip(0),
        'nb_ope_reseau_jcb'  : np.random.normal(4, 3, n_anomalie).clip(0),
        'nb_ope_reseau_postepargne'  : np.random.normal(5, 3, n_anomalie).clip(0),
        'nb_ope_reseau_diners_et_discovery'  : np.random.normal(3, 2, n_anomalie).clip(0),
        'nb_ope_reseau_autres'  : np.random.normal(10, 5, n_anomalie).clip(0),
        'type_gab_e_i'        : np.random.choice(['Interne', 'Externe'], n_anomalie),
        'code_postal'         : np.random.choice(['75001','69001','13001'], n_anomalie),
        'longitude'           : np.random.uniform(-4.5, 8.2, n_anomalie),
        'latitude'            : np.random.uniform(42.3, 51.1, n_anomalie),
        'is_anomalie_reelle'  : 1
    })

    return pd.concat([df_normal, df_anomalie], ignore_index=True)

df = generer_dataset()

print(f'📊 Dataset chargé : {df.shape[0]} lignes × {df.shape[1]} colonnes')
print(f'   → {df["num_automate"].nunique()} automates distincts')
df.head()

## 4. 🧹 Prétraitement des Données

Avant toute modélisation, nous vérifions la qualité des données et créons des variables métier enrichies.

In [ ]:
# ── 4.1 Valeurs manquantes ─────────────────────────────────────────────────────
manquants = df.isnull().sum()
print('🔍 Valeurs manquantes par colonne :')
print(manquants[manquants > 0] if manquants.sum() > 0 else '   ✅ Aucune valeur manquante')

# ── 4.2 Incohérences ──────────────────────────────────────────────────────────
incoherences = [
    ('ret_nb < 0',            (df['ret_nb'] < 0).sum()),
    ('taux_capture > 100',    (df['taux_capture_pct'] > 100).sum()),
    ('ret_pct_nuit > 100',    (df['ret_pct_nuit'] > 100).sum()),
]
print('\n🔍 Incohérences logiques :')
for label, count in incoherences:
    status = '✅' if count == 0 else '⚠️'
    print(f'   {status} {label} : {count} lignes')

In [ ]:
# ── 4.3 Feature Engineering ───────────────────────────────────────────────────

# Intensité de retrait : retraits par jour actif estimé (mois = 30j)
df['intensite_retrait']     = df['ret_nb'] / 30

# Ratio captures sur retraits (plus sensible que le % brut)
df['ratio_capture']         = df['cap_nb'] / (df['ret_nb'] + 1)

# Concentration réseau : proportion du réseau dominant
df['nb_ope_total_reseau']   = df['nb_ope_reseau_franfinance'] + df['nb_ope_reseau_cb'] + df['nb_ope_reseau_trionis'] + df['nb_ope_reseau_ppl'] + df['nb_ope_reseau_mastercard'] + df['nb_ope_reseau_configona'] + df['nb_ope_reseau_interne'] + df['nb_ope_reseau_casino'] + df['nb_ope_reseau_accord'] + df['nb_ope_reseau_visa'] + df['nb_ope_reseau_cos'] + df['nb_ope_reseau_jcb'] + df['nb_ope_reseau_postepargne'] + df['nb_ope_reseau_diners_et_discovery'] + df['nb_ope_reseau_autres']
df['concentration_reseau']  = df[['nb_ope_reseau_franfinance', 'nb_ope_reseau_cb', 'nb_ope_reseau_trionis', 'nb_ope_reseau_ppl', 'nb_ope_reseau_mastercard', 'nb_ope_reseau_configona', 'nb_ope_reseau_interne', 'nb_ope_reseau_casino', 'nb_ope_reseau_accord', 'nb_ope_reseau_visa', 'nb_ope_reseau_cos', 'nb_ope_reseau_jcb', 'nb_ope_reseau_postepargne', 'nb_ope_reseau_diners_et_discovery', 'nb_ope_reseau_autres']].max(axis=1) \\
                              / (df['nb_ope_total_reseau'] + 1)

# Score d'activité nocturne et weekend combiné
df['score_horaires_atypiques'] = (df['ret_pct_nuit'] / 10) + (df['ret_pct_weekend'] / 22)

# Variabilité des montants (coefficient de variation)
df['cv_montant']            = df['ret_montant_stddev'] / (df['ret_montant_moyen'] + 1)

print('✅ Features enrichies créées :')
features_nouvelles = ['intensite_retrait','ratio_capture','concentration_reseau',
                       'score_horaires_atypiques','cv_montant']
print(df[features_nouvelles].describe().round(3))

## 5. 📊 Analyse Exploratoire (EDA)

### Que ressemble un GAB « normal » ?

Avant de détecter des anomalies, il faut comprendre la distribution normale du réseau.

In [ ]:
# ── 5.1 Distributions principales ────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Distribution des indicateurs clés du réseau GAB', fontsize=16, fontweight='bold', y=1.01)

variables = [
    ('ret_nb',           'Nombre de retraits / mois'),
    ('ret_montant_moyen','Montant moyen de retrait (€)'),
    ('taux_capture_pct', 'Taux de capture (%)'),
    ('ret_pct_nuit',     '% retraits nocturnes'),
    ('ret_pct_weekend',  '% retraits weekend'),
    ('ratio_capture',    'Ratio captures / retraits'),
]

for ax, (col, titre) in zip(axes.flatten(), variables):
    data_plot = df[col].dropna()
    ax.hist(data_plot, bins=40, color=COULEUR_NORMAL, alpha=0.7, edgecolor='white')
    ax.axvline(data_plot.mean(),   color='navy',  lw=2, ls='--', label=f'Moyenne : {data_plot.mean():.1f}')
    ax.axvline(data_plot.median(), color='green', lw=2, ls=':',  label=f'Médiane : {data_plot.median():.1f}')
    ax.set_title(titre, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Fréquence')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('distributions_gab.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 La plupart des GAB se concentrent autour de valeurs stables → c\'est le comportement NORMAL.')

In [ ]:
# ── 5.2 Boxplots par type GAB ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Comportement par type de GAB (Interne vs Externe)', fontsize=14, fontweight='bold')

metriques_box = [
    ('ret_montant_moyen', 'Montant moyen (€)'),
    ('taux_capture_pct',  'Taux capture (%)'),
    ('ret_pct_nuit',      '% retraits nocturnes'),
]

for ax, (col, titre) in zip(axes, metriques_box):
    sns.boxplot(data=df, x='type_gab_e_i', y=col, ax=ax,
                palette={'Interne': COULEUR_NORMAL, 'Externe': COULEUR_ACCENT},
                width=0.5, flierprops={'marker':'o','alpha':0.4})
    ax.set_title(titre, fontweight='bold')
    ax.set_xlabel('Type de GAB')
    ax.set_ylabel('')

plt.tight_layout()
plt.savefig('boxplots_type_gab.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Les GAB Internes (en agence) ont des profils différents des GAB Externes → à comparer séparément.')

In [ ]:
# ── 5.3 Scatter : volume vs montants ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Relations entre indicateurs clés', fontsize=14, fontweight='bold')

axes[0].scatter(df['ret_nb'], df['ret_montant_total']/1000,
                alpha=0.4, c=COULEUR_NORMAL, edgecolors='none', s=30)
axes[0].set_xlabel('Nombre de retraits / mois')
axes[0].set_ylabel('Montant total (k€)')
axes[0].set_title('Volume vs Montant total')

axes[1].scatter(df['ret_nb'], df['taux_capture_pct'],
                alpha=0.4, c=COULEUR_NORMAL, edgecolors='none', s=30)
axes[1].set_xlabel('Nombre de retraits / mois')
axes[1].set_ylabel('Taux de capture (%)')
axes[1].set_title('Volume vs Taux de capture')

plt.tight_layout()
plt.savefig('scatter_indicateurs.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5.4 Heatmap de corrélation ─────────────────────────────────────────────────
features_corr = ['ret_nb','ret_montant_moyen','ret_montant_total',
                 'taux_capture_pct','ret_pct_nuit','ret_pct_weekend',
                 'ratio_capture','concentration_reseau','cv_montant']

corr_matrix = df[features_corr].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
plt.title('Matrice de corrélation — Indicateurs GAB', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('heatmap_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Des corrélations fortes entre variables peuvent indiquer un comportement cohérent (normal) ou anormal.')

## 6. 📅 Analyse Temporelle

### Le comportement des GAB évolue-t-il dans le temps ?

La **saisonnalité** est un facteur clé : une hausse d'activité en décembre est normale.  
Ce qui n'est **pas** normal : une hausse isolée sur un seul automate, hors saison.

In [ ]:
# ── 6.1 Agrégation mensuelle réseau ───────────────────────────────────────────
df_temps = df.groupby(['annee','mois']).agg(
    ret_nb_moyen        = ('ret_nb',            'mean'),
    montant_moyen       = ('ret_montant_moyen',  'mean'),
    taux_capture_moyen  = ('taux_capture_pct',   'mean'),
    pct_nuit_moyen      = ('ret_pct_nuit',       'mean'),
).reset_index()

df_temps['periode'] = pd.to_datetime(
    df_temps['annee'].astype(str) + '-' + df_temps['mois'].astype(str).str.zfill(2)
)
df_temps = df_temps.sort_values('periode')

# ── 6.2 Graphiques temporels ──────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
fig.suptitle('Évolution mensuelle du réseau GAB', fontsize=14, fontweight='bold')

labels_mois = ['Jan','Fév','Mar','Avr','Mai','Jun','Jul','Aoû','Sep','Oct','Nov','Déc']

for ax, (col, titre, couleur) in zip(axes, [
    ('ret_nb_moyen',       'Nombre moyen de retraits / GAB',      COULEUR_NORMAL),
    ('taux_capture_moyen', 'Taux moyen de capture (%)',           COULEUR_ANOMALIE),
    ('montant_moyen',      'Montant moyen de retrait (€)',        COULEUR_ACCENT),
]):
    ax.plot(df_temps['periode'], df_temps[col], marker='o', lw=2.5,
            color=couleur, markersize=6, markerfacecolor='white', markeredgewidth=2)
    ax.fill_between(df_temps['periode'], df_temps[col], alpha=0.1, color=couleur)
    ax.set_ylabel(titre, fontsize=10)
    ax.grid(axis='y', alpha=0.4)

    # Annotation décembre (pic naturel)
    dec_rows = df_temps[df_temps['mois'] == 12]
    for _, row in dec_rows.iterrows():
        ax.annotate('Déc.\n(Fêtes)', xy=(row['periode'], row[col]),
                    xytext=(10, 10), textcoords='offset points',
                    fontsize=7, color='grey',
                    arrowprops={'arrowstyle':'->', 'color':'grey', 'lw':0.8})

axes[-1].set_xlabel('Période')
plt.tight_layout()
plt.savefig('evolution_temporelle.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Un pic en décembre est normal. Un pic isolé sur un seul GAB est suspect.')

## 7. 🗺️ Analyse Géographique

La localisation d'un GAB influence fortement son comportement.  
Un GAB en centre-ville traite naturellement plus de retraits qu'un GAB rural.  
La carte nous permet de **contextualiser** chaque anomalie géographiquement.

In [ ]:
# ── 7.1 Carte interactive Plotly ──────────────────────────────────────────────
df_geo = df.groupby('num_automate').agg(
    longitude          = ('longitude',        'first'),
    latitude           = ('latitude',         'first'),
    ret_nb_moyen       = ('ret_nb',            'mean'),
    taux_capture_moyen = ('taux_capture_pct',  'mean'),
    type_gab           = ('type_gab_e_i',      'first'),
    code_postal        = ('code_postal',       'first'),
).reset_index()

fig_map = px.scatter_mapbox(
    df_geo,
    lat='latitude', lon='longitude',
    color='taux_capture_moyen',
    size='ret_nb_moyen',
    size_max=20,
    color_continuous_scale=['#2196F3', '#FF9800', '#F44336'],
    hover_name='num_automate',
    hover_data={'taux_capture_moyen': ':.2f', 'ret_nb_moyen': ':.0f', 'type_gab': True},
    zoom=5,
    center={'lat': 46.8, 'lon': 2.3},
    mapbox_style='carto-positron',
    title='🗺️ Réseau GAB — Intensité de retrait et taux de capture',
    labels={'taux_capture_moyen': 'Taux capture (%)', 'ret_nb_moyen': 'Nb retraits moyen'},
    height=550
)
fig_map.update_layout(margin={'r':0,'t':50,'l':0,'b':0})
fig_map.show()
print('💡 Les points rouges (taux de capture élevé) méritent une attention particulière.')

## 8. 🤖 Modélisation — Détection d'Anomalies

### Pourquoi l'Isolation Forest ?

| Modèle | Principe | Adapté ici ? |
|---|---|---|
| **Isolation Forest** | Isole les points rares par partitions aléatoires | ✅ Oui — rapide, robuste, interprétable |
| Local Outlier Factor | Densité locale vs voisins | ⚠️ Lent sur grands datasets |
| One-Class SVM | Frontière autour des normaux | ⚠️ Difficile à paramétrer |
| Seuils statistiques | z-score, IQR | ⚠️ Univarié, ne capte pas les interactions |

**Isolation Forest** est notre choix car :
- Il gère bien les **données multivariées** (50+ colonnes)
- Il est **rapide** sur de grands volumes
- Il produit un **score d'anomalie** continu (pas seulement un label)
- Il est **intuitif** : un point difficile à isoler = normal, facile à isoler = anormal

In [ ]:
# ── 8.1 Sélection des features pour le modèle ─────────────────────────────────
FEATURES_MODELE = [
    'ret_nb',
    'ret_montant_moyen',
    'ret_montant_max',
    'ret_montant_stddev',
    'ret_montant_total',
    'taux_capture_pct',
    'ret_pct_nuit',
    'ret_pct_weekend',
    'intensite_retrait',
    'ratio_capture',
    'concentration_reseau',
    'score_horaires_atypiques',
    'cv_montant',
    'nb_ope_reseau_franfinance',
    'nb_ope_reseau_cb',
    'nb_ope_reseau_trionis',
    'nb_ope_reseau_ppl',
    'nb_ope_reseau_mastercard',
    'nb_ope_reseau_configona',
    'nb_ope_reseau_interne',
    'nb_ope_reseau_casino',
    'nb_ope_reseau_accord',
    'nb_ope_reseau_visa',
    'nb_ope_reseau_cos',
    'nb_ope_reseau_jcb',
    'nb_ope_reseau_postepargne',
    'nb_ope_reseau_diners_et_discovery',
    'nb_ope_reseau_autres',
]

X = df[FEATURES_MODELE].fillna(0)

# ── 8.2 Normalisation ─────────────────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'✅ Matrice features : {X_scaled.shape[0]} lignes × {X_scaled.shape[1]} colonnes')

In [ ]:
# ── 8.3 Entraînement Isolation Forest ─────────────────────────────────────────
# contamination = proportion estimée d'anomalies dans le réseau (~5%)
modele = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)

df['score_anomalie'] = modele.fit_predict(X_scaled)  # -1 = anomalie, +1 = normal
df['score_if']       = modele.score_samples(X_scaled)  # plus bas = plus anormal

# Normalisation du score en [0, 1] pour lisibilité métier
score_min, score_max = df['score_if'].min(), df['score_if'].max()
df['score_risque'] = 1 - (df['score_if'] - score_min) / (score_max - score_min)
# score_risque proche de 1 = très anormal

df['est_anomalie'] = (df['score_anomalie'] == -1).astype(int)

n_anomalies = df['est_anomalie'].sum()
print(f'🚨 GAB atypiques détectés : {n_anomalies} / {len(df)} ({n_anomalies/len(df)*100:.1f}%)')

In [ ]:
# ── 8.4 Visualisation du score d'anomalie ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribution des scores d\'anomalie — Isolation Forest', fontsize=14, fontweight='bold')

# Histogramme des scores
ax = axes[0]
ax.hist(df[df['est_anomalie']==0]['score_risque'], bins=40,
        color=COULEUR_NORMAL, alpha=0.7, label='GAB normaux')
ax.hist(df[df['est_anomalie']==1]['score_risque'], bins=20,
        color=COULEUR_ANOMALIE, alpha=0.8, label='GAB atypiques')
ax.set_xlabel('Score de risque (0 = normal, 1 = très atypique)')
ax.set_ylabel('Nombre de GAB')
ax.set_title('Séparation normaux / atypiques')
ax.legend()

# Scatter retraits vs score risque
ax2 = axes[1]
couleurs = [COULEUR_ANOMALIE if a==1 else COULEUR_NORMAL for a in df['est_anomalie']]
ax2.scatter(df['ret_nb'], df['score_risque'], c=couleurs, alpha=0.5, s=20)
ax2.set_xlabel('Nombre de retraits / mois')
ax2.set_ylabel('Score de risque')
ax2.set_title('Retraits vs Score de risque')
patch_n = mpatches.Patch(color=COULEUR_NORMAL,   label='Normal')
patch_a = mpatches.Patch(color=COULEUR_ANOMALIE, label='Atypique')
ax2.legend(handles=[patch_n, patch_a])

plt.tight_layout()
plt.savefig('scores_anomalie.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. 🔎 Explication des Anomalies Détectées

### Pourquoi ces GAB sont-ils atypiques ?

C'est **la partie la plus importante** pour convaincre les experts métier.  
Chaque anomalie doit être **expliquée en langage métier**.

In [ ]:
# ── 9.1 Top 10 GAB les plus atypiques ─────────────────────────────────────────
top_anomalies = df[df['est_anomalie']==1].nlargest(10, 'score_risque')[
    ['num_automate','score_risque','ret_nb','ret_montant_moyen',
     'taux_capture_pct','ret_pct_nuit','ret_pct_weekend','type_gab_e_i']
].reset_index(drop=True)

top_anomalies.index += 1
top_anomalies.columns = ['Automate','Score risque','Nb retraits',
                          'Montant moyen (€)','Taux capture (%)','% Nuit','% Weekend','Type']

# Formatage
styled = top_anomalies.style \
    .background_gradient(subset=['Score risque'], cmap='Reds') \
    .background_gradient(subset=['Taux capture (%)'], cmap='Oranges') \
    .format({'Score risque': '{:.3f}', 'Taux capture (%)': '{:.2f}',
             'Montant moyen (€)': '{:.0f}', '% Nuit': '{:.1f}', '% Weekend': '{:.1f}'})

print('🚨 TOP 10 des GAB les plus atypiques :')
display(styled)

In [ ]:
# ── 9.2 Profil d'un GAB atypique vs le réseau ─────────────────────────────────
# Sélection du GAB le plus atypique
gab_anomalie = df[df['est_anomalie']==1].nlargest(1, 'score_risque').iloc[0]
gab_id = gab_anomalie['num_automate']

# Moyennes réseau (normaux)
moyennes_normaux = df[df['est_anomalie']==0][FEATURES_MODELE[:8]].mean()
valeurs_anomalie = df[df['num_automate']==gab_id][FEATURES_MODELE[:8]].mean()

# Calcul des écarts en % par rapport à la moyenne
ecarts = ((valeurs_anomalie - moyennes_normaux) / moyennes_normaux * 100).clip(-200, 500)

labels_fr = {
    'ret_nb'             : 'Nb retraits',
    'ret_montant_moyen'  : 'Montant moyen',
    'ret_montant_max'    : 'Montant max',
    'ret_montant_stddev' : 'Écart-type montant',
    'ret_montant_total'  : 'Montant total',
    'taux_capture_pct'   : 'Taux capture',
    'ret_pct_nuit'       : '% Nuit',
    'ret_pct_weekend'    : '% Weekend',
}

fig, ax = plt.subplots(figsize=(12, 5))
couleurs_barres = [COULEUR_ANOMALIE if v > 0 else COULEUR_NORMAL for v in ecarts.values]
bars = ax.barh([labels_fr.get(c, c) for c in ecarts.index], ecarts.values,
               color=couleurs_barres, alpha=0.85, edgecolor='white')
ax.axvline(0, color='black', lw=1.5)
ax.set_xlabel('Écart par rapport à la moyenne réseau (%)')
ax.set_title(f'🔍 Profil de {gab_id} vs moyenne réseau\n(rouge = supérieur à la normale, bleu = inférieur)',
             fontsize=13, fontweight='bold')

# Annotations valeur absolue
for bar, val in zip(bars, ecarts.values):
    x = bar.get_width()
    ax.text(x + (5 if x >= 0 else -5), bar.get_y() + bar.get_height()/2,
            f'{val:+.0f}%', va='center', ha='left' if x >= 0 else 'right', fontsize=9)

plt.tight_layout()
plt.savefig('profil_anomalie.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n📋 Interprétation métier de {gab_id} :')
print(f'   → Taux de capture : {gab_anomalie["taux_capture_pct"]:.1f}%  (réseau : {moyennes_normaux["taux_capture_pct"]:.1f}%)')
print(f'   → Montant moyen   : {gab_anomalie["ret_montant_moyen"]:.0f}€  (réseau : {moyennes_normaux["ret_montant_moyen"]:.0f}€)')
print(f'   → Activité nocturne : {gab_anomalie["ret_pct_nuit"]:.1f}%  (réseau : {moyennes_normaux["ret_pct_nuit"]:.1f}%)')

In [ ]:
# ── 9.3 Radar chart : comparaison GAB atypique vs réseau ──────────────────────
categories = ['Nb retraits', 'Montant moyen', 'Taux capture',
              '% Nuit', '% Weekend', 'Ratio capture']
cols_radar  = ['ret_nb','ret_montant_moyen','taux_capture_pct',
               'ret_pct_nuit','ret_pct_weekend','ratio_capture']

# Normalisation min-max pour le radar
def normaliser_radar(valeurs, ref_min, ref_max):
    return [(v - mn) / (mx - mn + 1e-9) for v, mn, mx in zip(valeurs, ref_min, ref_max)]

ref_min = df[cols_radar].quantile(0.05).values
ref_max = df[cols_radar].quantile(0.95).values

vals_anomalie = normaliser_radar(df[df['num_automate']==gab_id][cols_radar].mean().values, ref_min, ref_max)
vals_normal   = normaliser_radar(df[df['est_anomalie']==0][cols_radar].mean().values, ref_min, ref_max)

fig_radar = go.Figure()
for vals, nom, couleur in [
    (vals_anomalie, f'{gab_id} (Atypique)', COULEUR_ANOMALIE),
    (vals_normal,   'Moyenne réseau (Normal)', COULEUR_NORMAL),
]:
    fig_radar.add_trace(go.Scatterpolar(
        r=vals + [vals[0]],
        theta=categories + [categories[0]],
        fill='toself',
        name=nom,
        line_color=couleur,
        fillcolor=couleur,
        opacity=0.35
    ))

fig_radar.update_layout(
    polar={'radialaxis': {'visible': True, 'range': [0, 1]}},
    title=f'Radar — {gab_id} vs Moyenne réseau',
    height=500
)
fig_radar.show()
print('💡 Le GAB atypique dépasse significativement la zone normale sur plusieurs axes simultanément.')

In [ ]:
# ── 9.4 Carte des anomalies détectées ─────────────────────────────────────────
df_geo2 = df.groupby('num_automate').agg(
    longitude     = ('longitude',    'first'),
    latitude      = ('latitude',     'first'),
    score_risque  = ('score_risque', 'mean'),
    est_anomalie  = ('est_anomalie', 'max'),
    ret_nb_moyen  = ('ret_nb',       'mean'),
    taux_capture  = ('taux_capture_pct','mean'),
).reset_index()

df_geo2['statut'] = df_geo2['est_anomalie'].map({0: 'Normal', 1: 'Atypique'})

fig_anom_map = px.scatter_mapbox(
    df_geo2,
    lat='latitude', lon='longitude',
    color='statut',
    color_discrete_map={'Normal': COULEUR_NORMAL, 'Atypique': COULEUR_ANOMALIE},
    size='score_risque',
    size_max=18,
    hover_name='num_automate',
    hover_data={'taux_capture': ':.2f', 'ret_nb_moyen': ':.0f', 'score_risque': ':.3f'},
    zoom=5,
    center={'lat': 46.8, 'lon': 2.3},
    mapbox_style='carto-positron',
    title='🗺️ GAB Normaux vs Atypiques — Vue géographique',
    height=550
)
fig_anom_map.update_layout(margin={'r':0,'t':50,'l':0,'b':0})
fig_anom_map.show()

## 10. ✅ Validation du Modèle

### Comment évaluer un modèle sans étiquettes ?

Sans vérité terrain, nous utilisons 3 méthodes de validation alternatives :

1. **Validation statistique** : les anomalies détectées sont-elles vraiment aux extrêmes des distributions ?
2. **Analyse des cas extrêmes** : les GAB avec z-score > 3 sont-ils bien détectés ?
3. **Plausibilité métier** : les explications correspondent-elles à des situations connues ?

In [ ]:
# ── 10.1 Validation statistique ───────────────────────────────────────────────
print('📊 Comparaison statistique : Normaux vs Atypiques')
print('=' * 65)

stats_comp = df.groupby('est_anomalie')[[
    'ret_nb','ret_montant_moyen','taux_capture_pct','ret_pct_nuit'
]].mean().round(2)

stats_comp.index = ['🟢 Normaux', '🔴 Atypiques']
stats_comp.columns = ['Nb retraits', 'Montant moyen (€)', 'Taux capture (%)', '% Nuit']

print(stats_comp.to_string())
print()

# Ratios
ratios = (stats_comp.loc['🔴 Atypiques'] / stats_comp.loc['🟢 Normaux']).round(2)
print('📈 Ratios Atypiques / Normaux :')
for col, ratio in ratios.items():
    indicateur = '🔴 Anormal' if ratio > 1.5 or ratio < 0.5 else '🟡 Notable' if ratio > 1.2 else '🟢 Proche'
    print(f'   {indicateur}  {col} : ×{ratio}')

In [ ]:
# ── 10.2 Validation par z-scores (règle des 3σ) ───────────────────────────────
cols_zscore = ['ret_nb','ret_montant_moyen','taux_capture_pct','ret_pct_nuit']

for col in cols_zscore:
    mu  = df[col].mean()
    sig = df[col].std()
    df[f'zscore_{col}'] = (df[col] - mu) / (sig + 1e-9)

df['nb_zscore_extreme'] = sum(
    (df[f'zscore_{c}'].abs() > 3).astype(int) for c in cols_zscore
)

# Taux de détection croisée
detectes_zscore    = df[df['nb_zscore_extreme'] >= 1]['est_anomalie'].mean()
non_detectes_model = df[(df['est_anomalie']==1) & (df['nb_zscore_extreme']==0)].shape[0]

print(f'🔬 Validation croisée :')
print(f'   → {detectes_zscore*100:.1f}% des cas extrêmes (z>3) sont classés atypiques par le modèle')
print(f'   → {non_detectes_model} anomalies détectées par le modèle MAIS non visibles par z-score simple')
print(f'     (→ ce sont les anomalies subtiles, invisibles à l\'œil nu 🎯)')

In [ ]:
# ── 10.3 Réduction dimensionnelle PCA pour visualisation ──────────────────────
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

df['pca1'] = X_pca[:, 0]
df['pca2'] = X_pca[:, 1]

fig, ax = plt.subplots(figsize=(10, 7))

for label, mask, couleur, taille, alpha in [
    ('GAB Normaux',   df['est_anomalie']==0, COULEUR_NORMAL,   20, 0.35),
    ('GAB Atypiques', df['est_anomalie']==1, COULEUR_ANOMALIE, 80, 0.85),
]:
    ax.scatter(df[mask]['pca1'], df[mask]['pca2'],
               c=couleur, s=taille, alpha=alpha, label=label,
               edgecolors='white' if taille > 30 else 'none', linewidths=0.5)

ax.set_xlabel(f'Composante principale 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'Composante principale 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title('Visualisation PCA — Séparation Normaux vs Atypiques', fontsize=13, fontweight='bold')
ax.legend(markerscale=1.5, fontsize=11)

plt.tight_layout()
plt.savefig('pca_anomalies.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Les points rouges (atypiques) se situent bien aux marges du nuage principal → validation visuelle.')

## 11. 📖 Synthèse — Ce que le Modèle a Découvert

### Conclusions pour les équipes métier

In [ ]:
# ── 11.1 Tableau de bord synthèse ─────────────────────────────────────────────
n_total     = len(df)
n_anom      = df['est_anomalie'].sum()
n_normal_d  = n_total - n_anom
score_moy   = df[df['est_anomalie']==1]['score_risque'].mean()

print('=' * 60)
print('  📋 SYNTHÈSE — DÉTECTION D\'ANOMALIES GAB')
print('=' * 60)
print(f'  🏧 Total automates analysés      : {n_total}')
print(f'  🟢 GAB au comportement normal    : {n_normal_d} ({n_normal_d/n_total*100:.1f}%)')
print(f'  🔴 GAB au comportement atypique  : {n_anom} ({n_anom/n_total*100:.1f}%)')
print(f'  📊 Score de risque moyen (atyp.) : {score_moy:.3f} / 1.000')
print('=' * 60)
print()
print('  COMPORTEMENT NORMAL DU RÉSEAU :')
normaux = df[df['est_anomalie']==0]
print(f'  • Volume moyen      : {normaux["ret_nb"].mean():.0f} retraits/mois')
print(f'  • Montant moyen     : {normaux["ret_montant_moyen"].mean():.0f} €')
print(f'  • Taux capture moy. : {normaux["taux_capture_pct"].mean():.2f}%')
print(f'  • Activité nocturne : {normaux["ret_pct_nuit"].mean():.1f}%')
print()
print('  SIGNAUX D\'ALERTE LES PLUS FRÉQUENTS :')
print('  🔴 Taux de capture anormalement élevé (fraude / panne)')
print('  🔴 Montants moyens hors norme (opérations inhabituelles)')
print('  🔴 Activité nocturne excessive (comportement suspect)')
print('  🔴 Concentration réseau atypique (anomalie de flux cartes)')
print('=' * 60)

In [ ]:
# ── 11.2 Graphique final : les signaux d'alerte par fréquence ─────────────────
anom_df = df[df['est_anomalie']==1]
norm_df = df[df['est_anomalie']==0]

signaux = {
    'Taux capture > 2×moy.':       (anom_df['taux_capture_pct']   > norm_df['taux_capture_pct'].mean()*2).mean()*100,
    'Montant moyen > 2×moy.':      (anom_df['ret_montant_moyen']  > norm_df['ret_montant_moyen'].mean()*2).mean()*100,
    '% Nuit > 2×moy.':             (anom_df['ret_pct_nuit']       > norm_df['ret_pct_nuit'].mean()*2).mean()*100,
    '% Weekend > 2×moy.':          (anom_df['ret_pct_weekend']    > norm_df['ret_pct_weekend'].mean()*2).mean()*100,
    'Volume retraits hors norme':   (anom_df['ret_nb'].abs()       > norm_df['ret_nb'].mean()*2).mean()*100,
}

signaux_series = pd.Series(signaux).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(signaux_series.index, signaux_series.values,
               color=COULEUR_ANOMALIE, alpha=0.8, edgecolor='white')

for bar, val in zip(bars, signaux_series.values):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{val:.0f}% des atypiques', va='center', fontsize=10)

ax.set_xlabel('% des GAB atypiques présentant ce signal')
ax.set_title('Principaux signaux d\'alerte dans les GAB atypiques', fontsize=13, fontweight='bold')
ax.set_xlim(0, 110)
plt.tight_layout()
plt.savefig('signaux_alerte.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 11.3 Export de la liste des GAB à investiguer ─────────────────────────────
liste_investigation = df[df['est_anomalie']==1][
    ['num_automate','score_risque','ret_nb','ret_montant_moyen',
     'taux_capture_pct','ret_pct_nuit','ret_pct_weekend','type_gab_e_i','code_postal']
].sort_values('score_risque', ascending=False).reset_index(drop=True)

liste_investigation.index += 1

# Décommenter pour exporter
# liste_investigation.to_csv('gab_a_investiguer.csv', index=True)
# print('✅ Fichier exporté : gab_a_investiguer.csv')

print(f'📋 {len(liste_investigation)} GAB recommandés pour investigation :')
display(liste_investigation.head(15).style
    .background_gradient(subset=['score_risque'], cmap='Reds')
    .format({'score_risque': '{:.3f}', 'taux_capture_pct': '{:.2f}',
             'ret_montant_moyen': '{:.0f}'}))

## 12. 📌 Conclusion & Recommandations

---

### Ce que le modèle apporte

| Ce que vous faisiez avant | Ce que le modèle permet maintenant |
|---|---|
| Surveillance manuelle par seuil | Détection automatique multi-dimensionnelle |
| Anomalies évidentes seulement | Détection de signaux faibles combinés |
| Analyse rétrospective | Alerte mensuelle systématique |
| Subjectif / expertise individuelle | Reproductible et documenté |

---

### Prochaines étapes recommandées

1. **Validation métier** : soumettre la liste des GAB détectés aux experts pour qualification (vrai/faux positif)
2. **Enrichissement** : intégrer des données de contexte (incidents déclarés, interventions techniques)
3. **Mise en production** : exécution mensuelle automatique dans Dataiku
4. **Amélioration continue** : avec les retours métier, passer progressivement à un modèle semi-supervisé

---

> *« Un modèle n'est pas là pour remplacer l'expertise métier, mais pour l'amplifier.  
> Il voit ce que l'œil ne peut pas voir sur 500 automates simultanément. »*

---